DEBUGGING CON TENSORBOARD: VISUALIZZARE L'APPRENDIMENTO

Tensorboard non serve solo a vedere grafici di loss e accuracy.
La parte veramente interessante è il debugging del training.
Il vero problema nel deep learning è capire se il modello sta imparando qualcosa di sensato.
Quando lanci modello.fit(...) non sai se sta overfittando, se i gradienti esplodono, i neurono sono morti, il layer non apprende, la rete sta saturando, ecc.
Senza strumenti di debugging non lo sai
TensoBoard ti dice questo

Non vedi solo se il modello sta imparando ma come e perchè.

ECOSISTEM DEI LOG
La pipeline di osservabilità
TensorBoard non è integrato dentro la rete neurale ma un'entità separate, non è semplicemente un generatore di grafici, ma una suite di visualizzazone disaccoppiata. Essa legge file binari serializzati, chiamati 'event files', prodotti durante l'addestramento.
Questa architettura permette al modello di correre alla massima velocità (GPU) senza essere rallentato dal dover visualizzare grafici in tempo reale (gestiti tramite event files)

- LogDir: la cartella radice dove vengono salvate le diverse sessioni di addestramento, fondamentale per il confronto tra esperimenti
- Summary Writers: oggetti che trasformano tensori e variabili Python in formati compatibili con i protocolli di comunicazione di TensorBoard.
- Scalars: il formato più comune per tracciare metriche scalari che variano nel tempo, come la perdita e l'accuratezza.
- Asincronia: la scrittura dei log avviene solitamente su thread separati per non bloccare il flusso dei gradienti durante il calcolo. Il modello lancia i dati al logger, e continua il suo lavoro.

Gestione delle Run
Per non perderci tra centinaio di esperimenti diversi.
Usare il timestamping per nominare le cartelle non è solo ordine, è soppravivenza, permette di distinguere automaticamente diverse configurazioni di iperparametri all'interno della stessa vista.
Frequenza dei log
Scrivere log troppo frequentemente può saturare il bus di memoria; è necessario bilanciare la granularità dei dati con l'efficienza del sistema.
Persistenza
I file di log rimangono disponibili anche dopo la chiusura dello script, agendo come un archivio storico dello sviluppo del progetto di ricerca

Protocollo Buffers
Il formato di serializzazione
Sotto il cofano, TensorBoard utilizza 'Protocolli Buffers' di Google per garantire che i dati siano compatti e veloci da leggere, indipendentemente dalla piattaforma hardware.
Invece di usare file di testo pesanti, usiamo un formato binario compatto. Questo gatantisce che TensorBoard legga i dati alla stessa velocità indipendentemnete dal dispositivo di visualizzazione (pc, tablet, ecc)

Analisi del Grafo e dei Pesi
Oltre le semplici curve di loss
Le curve di Loss ci dicono se il modello sta andando bene, ma non ci dicono sulla sulla sua architettura interna.

- Graph View: rappresenta i tensori come archi e le operazioni come nodi, permettendo di verificare la corretezza della pipeline dati.
- Histograms: visualizzano la distribuzione dei valori dei pesi in ogni layer, aiutando a notare saturazioni nelle funzioni di attivazione
- Distributions: mostrano l'andamento dei parametri nel tempo, evidenziando se i pesi stanno cambiando o se la rete è in stallo.
- Weights vs Gradients: è possibile loggare anche l'intensità dei gradienti per identificare quale parte della rete è più difficile da addestrare

Analisi delle Attivazioni
Se gli istogrammi mostrano pesi tutti vicini allo zero o ai limiti della funzione di attivazione, potremmo essere di fronte a  una ReLU morente o una Sigmoide satura.
Utilizzando i 'name scopes' nel codice, possiamo raggruppare i nodi del grafo in blocchi logici (es. layer1, Optimizer) rendendo la dashboard ordinata.
TensorBoard permette anche di visualizzare i filtri delle CNN o gli input trasformati, offrendo una vista direta su cosa la rete sta 'vedendo'

Il Grafo Computazionale Live
Validazione della topologia
Ispezionare il grafo è il modo più rapido per scoprire se un layer è stato accidentalmente isolato dal flusso dei gradienti o se le dimensioni dei tensori sono incoerenti.
In PyTorch e TensorFlow, questa visualizzazione dinamica è fondamentale per modelli complessi con molteplici rami di input e output

Come attivare tutto questo?
La buona notizia è che non dobbiamo scrivere noi il sistema di loggin da zero.
Collegare TensorBoard al proprio workflow richiede poche righe di codice, sia che si utilizzi Keras con le sue callback, sia che si preferisca l'approccio manuale di PyTorch

SETEUP IN KERAS e PyTorch

In Keras è semplicissimo, basta una riga di codice per dire al modello, mentre studi scrivi in questa cartella. Utilizziamo una callaback
In PyTorch usiamo il SummaryWriter che ci sa un controllo più granulare permettendo di decidere quale variabile loggare a quando.
Quando lavoriamo in cloud, come per esempio Google Colab, useremo il Port Forwarding per visualizzare la dashboard tramite un tunnel sicuro, che porti la dsahboard dalla macchina remota in locale.

La Dashboard è un ambiente interattivo. 
- Filtraggio e Regex: è possibile utilizzare espressioni regolari nella barra di ricerca per isolare metriche specifiche o confrontare solo determinati esperimenti
- Smoothing: permette di ridurre il rumore nelle curve di loss, rendendo più visibile la tendenza generale dell'addestramento
- HParams Plugin: una sezione dedicata che permette di confrontare diverse combinazioni di iperparametri tramite grafici a coordinate parallele


In [2]:
#import sys
#!{sys.executable} -m pip install tensorboard

Cosa può debaggare:

1. LOSS E ACCURACY

Grafico tipico sano:
- training loss: scende
- validation loss: scende
Grafico problematico:
- training loss: scende
- validation loss: sale
Questo è overfitting

2. LEARNING RATE DUBIGGING

Problema frequente:
- Learning rate troppo alto: loss oscilla, training instabile, gradienti esplodono
- Learning rate troppo basso: training lentissimo, stuck
Con Loss del tipo: 0.8   5.4   0.2   9.1   0.1
Oscilla troppo, learning rate troppo agressivo
Con Loss del tipo: 0.8   0.799   0.798   0.797
Sei praticamente fermo, sembra lr troppo basso

3. HISTOGRAM DEBUGGING

Puoi vedere distribuzione pesi, bias, attivazioni
Con:
TensorBoard(
    log_dir="logs",
    histogram_freq=1
)
TensorBoard salva istogrammi, importante per vedere: exploding gradients, vanisching gradients, dead neurons 
Problema tipico ReLU: neuroni morti
Negli istogrammi vedi distribuzione schiacciata a zero

4. GRADIENT DEBUGGING

Se i gradienti esplodono o spariscono sono segnali di layer profondi non imparano (vanishing gradients), oppure loss Nan o training instabile

5. GRAPH DEBUGGING

TensorBoard può mostrare il grafico computazionale. Utileper capire architetture, verificare connessioni, debugging modelli complessi.
Puoi vedere il flusso dei tensori

6. EMBEDDING PROJECT

Per visualizzare embedding in 2D, 3D utile in NLP, clustering

7. PROFILE PERMORMANCE

TensorBoard può prifilare: CPU, GPU, memoria, tempi layer

8. DEBUGGING CUSTOM

Puoi scrivere metriche personalizzate

In [ ]:
import os
import datetime

# --- CONFIGURAZIONE AMBIENTE ---
# Disattiva i log di sistema non necessari di TensorFlow (info e warning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# Disattiva le ottimizzazioni oneDNN per evitare differenze di precisione numerica trascurabili
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow.keras import layers, models, Input

def main():
    # 1. PREPARAZIONE DATI
    print("Caricamento dati...")
    # Carichiamo il dataset MNIST (60.000 immagini di cifre scritte a mano)
    mnist = tf.keras.datasets.mnist
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    # NORMALIZZAZIONE: Trasformiamo i valori dei pixel da [0, 255] a [0, 1]
    # Questo aiuta la rete neurale a convergere (imparare) più velocemente
    x_train, x_test = x_train / 255.0, x_test / 255.0

    # 2. DEFINIZIONE DEL MODELLO (Architettura della Rete)
    # Creiamo un modello sequenziale (uno strato dopo l'altro)
    model = models.Sequential([
        # Definiamo l'input: immagini 28x28 pixel
        Input(shape=(28, 28), name="Input_Layer"),
        
        # Trasformiamo la matrice 28x28 in un vettore piatto di 784 elementi
        layers.Flatten(),
        
        # Strato denso (completamente connesso) con 128 neuroni
        # 'relu' è la funzione di attivazione standard per evitare la scomparsa del gradiente
        layers.Dense(128, activation='relu', name="Hidden_Layer_1"),
        
        # DROPOUT: Spegne casualmente il 20% dei neuroni durante il training
        # Serve a prevenire l'Overfitting (la rete non impara a memoria i dati)
        layers.Dropout(0.2, name="Regularization"),
        
        # Strato di output: 10 neuroni (uno per ogni cifra da 0 a 9)
        # 'softmax' trasforma l'output in probabilità (la somma di tutti i neuroni sarà 1)
        layers.Dense(10, activation='softmax', name="Output_Layer")  #con softmax posso gestire più classi in output
    ])

    # COMPILAZIONE: Definiamo come il modello deve imparare
    model.compile(
        optimizer='adam',                # Algoritmo di ottimizzazione (molto efficiente)
        loss='sparse_categorical_crossentropy', # Funzione di errore per classificazioni multi-classe
        metrics=['accuracy']             # Metrica per valutare la performance
    )

    # 3. CONFIGURAZIONE TENSORBOARD (Monitoraggio)
    # Creiamo una cartella specifica basata sul timestamp per ogni esecuzione
    #%load_ext tensorboard
    log_dir = os.path.join(os.getcwd(), "logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    
    # Il callback TensorBoard scriverà i log durante l'allenamento
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir, 
        histogram_freq=1, # Calcola la distribuzione dei pesi ad ogni epoca
        write_graph=True,  # Salva il grafico della struttura del modello
        write_images=True # Visualizza i pesi del modello come immagini in TensorBoard.
    )

    # 4. TRAINING (Allenamento)
    print(f"I log verranno salvati in: {log_dir}")
    model.fit(
        x_train, y_train, 
        epochs=20,                  # Numero di passaggi completi sui dati
        validation_data=(x_test, y_test), # Valuta la precisione su dati mai visti dopo ogni epoca
        callbacks=[tensorboard_callback]  # Attiva TensorBoard
    )

if __name__ == "__main__":
    main()

# per attivare la tensorboard: tensorboard --logdir="logs/fit"

Caricamento dati...
I log verranno salvati in: c:\Users\uberti\iCloudDrive\iCloudDrive\Barbara\EPICODE\PYTHON\MODULO 4\3_TensorFlow_PyTorch\logs\fit\20260514-172536
Epoch 1/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9106 - loss: 0.3049 - val_accuracy: 0.9583 - val_loss: 0.1414
Epoch 2/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9574 - loss: 0.1441 - val_accuracy: 0.9710 - val_loss: 0.0981
Epoch 3/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9668 - loss: 0.1092 - val_accuracy: 0.9711 - val_loss: 0.0893
Epoch 4/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9729 - loss: 0.0887 - val_accuracy: 0.9737 - val_loss: 0.0847
Epoch 5/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9766 - loss: 0.0758 - val_accuracy: 0.9754 - val_loss: 0.0799
Epoch 6/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9793 - loss: 0.0669 - val_accuracy: 0.9771 - val_loss: 0.0744
Epoch 7/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step